In [2]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

username = 'postgres'
password = 'saklani2003'
host = "localhost"
port = "5432"
database = "pizza_sales"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

In [3]:
query = pd.read_sql_query("""SELECT table_name FROM information_schema.tables 
WHERE table_schema = 'public' AND table_type = 'BASE TABLE'""", con= engine)
query

,table_name
0,order_details
1,pizzas
2,orders
3,pizza_types


In [4]:
for table in query['table_name']:
    print('-'*50, f'{table}','-'*50)
    print('count of records:', pd.read_sql(f"SELECT count(*) as count FROM {table}", con = engine)['count'].values[0])
    display(pd.read_sql(f"SELECT * FROM {table} limit 5", con=engine))

-------------------------------------------------- order_details --------------------------------------------------
count of records: 48620


,order_details_id,order_id,pizza_id,quantity
0,1,1,hawaiian_m,1
1,2,2,classic_dlx_m,1
2,3,2,five_cheese_l,1
3,4,2,ital_supr_l,1
4,5,2,mexicana_m,1


-------------------------------------------------- pizzas --------------------------------------------------
count of records: 96


,pizza_id,pizza_type_id,size,price
0,bbq_ckn_s,bbq_ckn,S,12.75
1,bbq_ckn_m,bbq_ckn,M,16.75
2,bbq_ckn_l,bbq_ckn,L,20.75
3,cali_ckn_s,cali_ckn,S,12.75
4,cali_ckn_m,cali_ckn,M,16.75


-------------------------------------------------- orders --------------------------------------------------
count of records: 21350


,order_id,date,time
0,1,2015-01-01,11:38:36
1,2,2015-01-01,11:57:40
2,3,2015-01-01,12:12:28
3,4,2015-01-01,12:16:31
4,5,2015-01-01,12:21:30


-------------------------------------------------- pizza_types --------------------------------------------------
count of records: 32


,pizza_type_id,name,category,ingredients
0,bbq_ckn,The Barbecue Chicken Pizza,Chicken,"Barbecued Chicken, Red Peppers, Green Peppers,..."
1,cali_ckn,The California Chicken Pizza,Chicken,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."
2,ckn_alfredo,The Chicken Alfredo Pizza,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A..."
3,ckn_pesto,The Chicken Pesto Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Spinach, Garli..."
4,southw_ckn,The Southwest Chicken Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Red Onions, Ja..."


In [5]:
order_details = pd.read_sql_query("select * from order_details", con=engine)
order_details

,order_details_id,order_id,pizza_id,quantity
0,1,1,hawaiian_m,1
1,2,2,classic_dlx_m,1
2,3,2,five_cheese_l,1
3,4,2,ital_supr_l,1
4,5,2,mexicana_m,1
...,...,...,...,...
48615,48616,21348,ckn_alfredo_m,1
48616,48617,21348,four_cheese_l,1
48617,48618,21348,napolitana_s,1
48618,48619,21349,mexicana_l,1


In [6]:
order_details.groupby('pizza_id')["quantity"].sum()

pizza_id
bbq_ckn_l         992
bbq_ckn_m         956
bbq_ckn_s         484
big_meat_s       1914
brie_carre_s      490
                 ... 
the_greek_xl      552
the_greek_xxl      28
veggie_veg_l      427
veggie_veg_m      635
veggie_veg_s      464
Name: quantity, Length: 91, dtype: int64

### As per the above code we get to know that each pizza_id contains different quantity purchased
Total number to pizza id are 91 


In [29]:
order_details['pizza_id'].nunique()

91

In [31]:
pizzas  = pd.read_sql_query("select * from pizzas", con=engine)
pizzas 

,pizza_id,pizza_type_id,size,price
0,bbq_ckn_s,bbq_ckn,S,12.75
1,bbq_ckn_m,bbq_ckn,M,16.75
2,bbq_ckn_l,bbq_ckn,L,20.75
3,cali_ckn_s,cali_ckn,S,12.75
4,cali_ckn_m,cali_ckn,M,16.75
...,...,...,...,...
91,spinach_fet_m,spinach_fet,M,16.00
92,spinach_fet_l,spinach_fet,L,20.25
93,veggie_veg_s,veggie_veg,S,12.00
94,veggie_veg_m,veggie_veg,M,16.00


pizza id is mirror of pizza_type_id plus size which make it primary key in this table
price of pizza wary for different sizes.

we need to creat summary table  from the given tables to do final analysis.


In [34]:
pizza_sales = pd.read_sql_query("""
        SELECT od.order_details_id,
        od.pizza_id,  
        p.pizza_type_id as pizza_name_id, 
        o.date as order_date,
        o.time as order_time,
		od.quantity,
        p.price,
        p.size as pizza_size,
        pt.category as pizza_category,
        pt.ingredients,
        pt.name as pizza_name
        
        FROM order_details od JOIN pizzas p ON od.pizza_id  = p.pizza_id
        JOIN orders o ON od.order_id = o.order_id
        JOIN pizza_types pt ON p.pizza_type_id = pt.pizza_type_id
        
        ORDER BY order_date""", engine)
pizza_sales

,order_details_id,pizza_id,pizza_name_id,order_date,order_time,quantity,price,pizza_size,pizza_category,ingredients,pizza_name
0,1,hawaiian_m,hawaiian,2015-01-01,11:38:36,1,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,classic_dlx_m,classic_dlx,2015-01-01,11:57:40,1,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,five_cheese_l,five_cheese,2015-01-01,11:57:40,1,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,ital_supr_l,ital_supr,2015-01-01,11:57:40,1,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,mexicana_m,mexicana,2015-01-01,11:57:40,1,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza
...,...,...,...,...,...,...,...,...,...,...,...
48615,48616,ckn_alfredo_m,ckn_alfredo,2015-12-31,21:23:10,1,16.75,M,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A...",The Chicken Alfredo Pizza
48616,48617,four_cheese_l,four_cheese,2015-12-31,21:23:10,1,17.95,L,Veggie,"Ricotta Cheese, Gorgonzola Piccante Cheese, Mo...",The Four Cheese Pizza
48617,48618,napolitana_s,napolitana,2015-12-31,21:23:10,1,12.00,S,Classic,"Tomatoes, Anchovies, Green Olives, Red Onions,...",The Napolitana Pizza
48618,48619,mexicana_l,mexicana,2015-12-31,22:09:54,1,20.25,L,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [36]:
pizza_sales['total_price'] = summ['quantity']*summ['price']

NameError: name 'summ' is not defined

In [38]:
pizza_sales

,order_details_id,pizza_id,pizza_name_id,order_date,order_time,quantity,price,pizza_size,pizza_category,ingredients,pizza_name
0,1,hawaiian_m,hawaiian,2015-01-01,11:38:36,1,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,classic_dlx_m,classic_dlx,2015-01-01,11:57:40,1,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,five_cheese_l,five_cheese,2015-01-01,11:57:40,1,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,ital_supr_l,ital_supr,2015-01-01,11:57:40,1,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,mexicana_m,mexicana,2015-01-01,11:57:40,1,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza
...,...,...,...,...,...,...,...,...,...,...,...
48615,48616,ckn_alfredo_m,ckn_alfredo,2015-12-31,21:23:10,1,16.75,M,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A...",The Chicken Alfredo Pizza
48616,48617,four_cheese_l,four_cheese,2015-12-31,21:23:10,1,17.95,L,Veggie,"Ricotta Cheese, Gorgonzola Piccante Cheese, Mo...",The Four Cheese Pizza
48617,48618,napolitana_s,napolitana,2015-12-31,21:23:10,1,12.00,S,Classic,"Tomatoes, Anchovies, Green Olives, Red Onions,...",The Napolitana Pizza
48618,48619,mexicana_l,mexicana,2015-12-31,22:09:54,1,20.25,L,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


we combine all the give data into into single table and also created new table from the given data so we can get insights

the new feature we added is total_price which we obtained by multiplaying quantity and price columns from order_details and pizzas tables.

We also arrange feature according to our need.

In [41]:
pizza_sales.dtypes

order_details_id      int64
pizza_id             object
pizza_name_id        object
order_date           object
order_time           object
quantity              int64
price               float64
pizza_size           object
pizza_category       object
ingredients          object
pizza_name           object
dtype: object

In [43]:
pizza_sales.isnull().sum()

order_details_id    0
pizza_id            0
pizza_name_id       0
order_date          0
order_time          0
quantity            0
price               0
pizza_size          0
pizza_category      0
ingredients         0
pizza_name          0
dtype: int64

In [59]:
pizza_sales['total_price'] = pizza_sales['quantity']*pizza_sales['price']

In [61]:
pizza_sales['total_price'].min()

9.75

In [63]:
pizza_sales['total_price'].max()

83.0

In [65]:
conn = psycopg2.connect(
    user = 'postgres',
    password = 'saklani2003',
    host = "localhost",
    database = "pizza_sales")
cursor = conn.cursor()

In [67]:
pizza_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_details_id  48620 non-null  int64  
 1   pizza_id          48620 non-null  object 
 2   pizza_name_id     48620 non-null  object 
 3   order_date        48620 non-null  object 
 4   order_time        48620 non-null  object 
 5   quantity          48620 non-null  int64  
 6   price             48620 non-null  float64
 7   pizza_size        48620 non-null  object 
 8   pizza_category    48620 non-null  object 
 9   ingredients       48620 non-null  object 
 10  pizza_name        48620 non-null  object 
 11  total_price       48620 non-null  float64
dtypes: float64(2), int64(2), object(8)
memory usage: 4.5+ MB


In [69]:
pizza_sales.columns

Index(['order_details_id', 'pizza_id', 'pizza_name_id', 'order_date',
       'order_time', 'quantity', 'price', 'pizza_size', 'pizza_category',
       'ingredients', 'pizza_name', 'total_price'],
      dtype='object')

In [71]:
cursor.execute("""CREATE TABLE pizza_sales (
	order_details_id INT,
    pizza_id SERIAL,
	pizza_name_id VARCHAR(35),
	order_date DATE,
	order_time TIME,
    quantity INT,
	price NUMERIC,
	pizza_size VARCHAR(5),
	pizza_category VARCHAR(35),
	ingredients VARCHAR(100),
	pizza_name VARCHAR(55),
    total_price NUMERIC
	);
    """
)

In [73]:
pd.read_sql_query("SELECT * FROM pizza_sales;", conn)

C:\Users\saklani\AppData\Local\Temp\ipykernel_10592\720361221.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("SELECT * FROM pizza_sales;", conn)


,order_details_id,pizza_id,pizza_name_id,order_date,order_time,quantity,price,pizza_size,pizza_category,ingredients,pizza_name,total_price


In [77]:
pizza_sales.to_sql('pizza_sales', engine, if_exists = "replace", index = False)

620

In [79]:
pd.read_sql_query("SELECT * FROM pizza_sales;", conn)

C:\Users\saklani\AppData\Local\Temp\ipykernel_10592\720361221.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("SELECT * FROM pizza_sales;", conn)


,order_details_id,pizza_id,pizza_name_id,order_date,order_time,quantity,price,pizza_size,pizza_category,ingredients,pizza_name,total_price
0,1,hawaiian_m,hawaiian,2015-01-01,11:38:36,1,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza,13.25
1,2,classic_dlx_m,classic_dlx,2015-01-01,11:57:40,1,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza,16.00
2,3,five_cheese_l,five_cheese,2015-01-01,11:57:40,1,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza,18.50
3,4,ital_supr_l,ital_supr,2015-01-01,11:57:40,1,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza,20.75
4,5,mexicana_m,mexicana,2015-01-01,11:57:40,1,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza,16.00
...,...,...,...,...,...,...,...,...,...,...,...,...
48615,48616,ckn_alfredo_m,ckn_alfredo,2015-12-31,21:23:10,1,16.75,M,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A...",The Chicken Alfredo Pizza,16.75
48616,48617,four_cheese_l,four_cheese,2015-12-31,21:23:10,1,17.95,L,Veggie,"Ricotta Cheese, Gorgonzola Piccante Cheese, Mo...",The Four Cheese Pizza,17.95
48617,48618,napolitana_s,napolitana,2015-12-31,21:23:10,1,12.00,S,Classic,"Tomatoes, Anchovies, Green Olives, Red Onions,...",The Napolitana Pizza,12.00
48618,48619,mexicana_l,mexicana,2015-12-31,22:09:54,1,20.25,L,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza,20.25
